## Observer JSON -> Phenopacket v2.0

Batches every real Observer JSON fixture in `tests/data/*_Sally_pretty.json`
(5 files: Apple, Blue, Charm, Diva, Eclair) through the real
`build_observer_phenopacket()` builder rather than hand-rolling the
pipeline inline. For each fetus this shows three layers together:

- **HPO features** (the builder's own output) - biometry, clinical
  impression, and fetal anatomy findings, each resolved to an HPO term.
- **LOINC-coded measurements** - the same biometry re-extracted as raw
  mm values with LOINC assay codes, hand-added on top since the builder
  doesn't emit `Measurement`s yet.
- **Dummy genomics scaffolding** - a VCF file + genomic interpretation
  attached to every fetus (`Apple_Sally.vcf`, reused across all cases
  since these are synthetic fixtures, not real per-patient sequencing
  data). Structural only - no VRS normalization, no ACMG calls.

Round-trips every Phenopacket through JSON to confirm it's schema-valid,
and saves each to `notebook_outputs/observer/`.

In [ ]:
import gzip
import json
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

from prenatalppkt.builders import build_observer_phenopacket
from prenatalppkt.etl.extractors import observer as observer_extractor
from prenatalppkt.genomics import (
    build_genomic_interpretation,
    build_vcf_file_entry,
    scan_vcf_file,
)
from prenatalppkt.hpo import HpoParser

print("=" * 80)
print("OBSERVER JSON -> Phenopacket v2.0")
print("Covers every real (non-gyn) Observer JSON fixture: HPO features (the")
print("builder), LOINC-coded raw measurements (hand-added on top), and a")
print("dummy genomics attachment (VCF file + interpretation) on every fetus.")
print("=" * 80)

# -----------------------------------------------------------------------------
# STEP 1: Load the fenominal HPO Concept Recognizer (text -> HPO, detects negation)
# -----------------------------------------------------------------------------
HP_JSON_GZ = Path("tests/data/hp.json.gz")
TMP_HP_JSON = Path("/tmp/hp_observer_cell.json")
with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
    with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
        f_out.write(f_in.read())
hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
print(f"HPO version: {hpo_parser.get_version()}")

DATA_DIR = Path("tests/data")
now_ts = Timestamp()
now_ts.FromDatetime(datetime.now(tz=timezone.utc))

# All Observer fixtures are synthetic ("Sally" placeholder patients), and the
# VCF fixtures are dummy variant lists too, not tied to any one patient - so
# the same VCF is reused across every fetus below rather than only attaching
# genomics where a same-named .vcf happens to exist.
DUMMY_VCF = DATA_DIR / "Apple_Sally.vcf"

OUTPUT_DIR = Path("notebook_outputs/observer")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------------------------------------------------------
# STEP 2: Loop over every real Observer JSON fixture
# -----------------------------------------------------------------------------
# Explicit list, not a glob: tests/data/ also has Gwen_Sally_pretty.json (a
# gyn exam fixture, matches the same *_Sally_pretty.json pattern but has an
# empty fetuses list - build_observer_phenopacket is for fetal exams only,
# gyn exams go through build_gyn_phenopacket instead, out of scope here.
OBSERVER_FIXTURES = ["Apple", "Blue", "Charm", "Diva", "Eclair"]
data_files = [DATA_DIR / f"{name}_Sally_pretty.json" for name in OBSERVER_FIXTURES]
print(f"\nFiles: {len(data_files)}")

for data_path in data_files:
    print("\n" + "-" * 80)
    print(f"Processing: {data_path.name}")
    print("-" * 80)

    raw = json.loads(data_path.read_text())
    accession_id = data_path.stem.replace("_pretty", "")

    # STEP 3: the builder does the real work - biometry, clinical impression,
    # fetal anatomy, dating all stitched into one Phenopacket per fetus.
    pps = build_observer_phenopacket(raw, hpo_parser, now_ts, accession_id=accession_id)
    print(f"Fetuses: {len(pps)}")

    # Re-run the extractor directly so we have the raw TermBins (with their
    # LOINC codes + raw mm values) to build Measurements from - the builder
    # itself only emits phenotypicFeatures, not measurements, today.
    term_bins_by_fetus = observer_extractor.extract_all_fetuses(raw)

    for pp in pps:
        fetus_number = int(pp.id.rsplit("-", 1)[-1])
        term_bins = term_bins_by_fetus.get(fetus_number, [])

        # STEP 4: hand-add LOINC-coded Measurements alongside the HPO features -
        # this is the "raw value" view next to the "clinical interpretation" view.
        measurements = [
            pps2.Measurement(
                assay=pps2.OntologyClass(id=tb.loinc_code, label=tb.loinc_label),
                value=pps2.Value(
                    quantity=pps2.Quantity(
                        unit=pps2.OntologyClass(id="UO:0000016", label="millimeter"),
                        value=tb.value_mm,
                    )
                ),
            )
            for tb in term_bins
            if tb.loinc_code
        ]
        pp.measurements.extend(measurements)

        # STEP 5: attach dummy genomics scaffolding (VCF file + interpretation).
        # Structural only - no VRS normalization, no ACMG calls, no real linkage
        # between these variants and this fetus's actual phenotype.
        variants = scan_vcf_file(DUMMY_VCF)
        pp.files.append(
            build_vcf_file_entry(
                DUMMY_VCF.resolve().as_uri(),
                attributes={"genomeAssembly": variants[0].genome_assembly},
            )
        )
        pp.interpretations.append(
            build_genomic_interpretation(
                variants,
                subject_id=pp.subject.id,
                interpretation_id=f"{pp.id}-genomic-interp-1",
            )
        )
        print(
            f"  {pp.id}: {len(pp.phenotypic_features)} features, "
            f"{len(measurements)} measurements, {len(variants)} dummy variants"
        )

        # STEP 6: round-trip through JSON to confirm the Phenopacket is schema-valid
        json_str = MessageToJson(pp)
        round_tripped = Parse(json_str, pps2.Phenopacket())
        assert round_tripped == pp

        out_path = OUTPUT_DIR / f"{pp.id}_phenopacket.json"
        out_path.write_text(json_str)

print("\nSaved to:", OUTPUT_DIR)
TMP_HP_JSON.unlink(missing_ok=True)


## ViewPoint HL7 -> Phenopacket v2.0

Batches every synthetic ViewPoint HL7 fixture through the real
`build_viewpoint_phenopacket()` builder, the same three-layer shape as
the Observer cell above (HPO features, LOINC measurements, dummy
genomics):

- `viewpoint_hl7_test.txt` - single fetus
- `viewpoint_hl7_twins_test.txt` - twins, two fetuses
- `viewpoint_hl7_full_exam_test.txt` - biometry + all 16 anatomy fields
- `viewpoint_hl7_anatomy_test.txt` - anatomy only, no biometry (produces
  0 fetuses - documented builder behavior, not an error, since fetus
  grouping is keyed off biometry OBX segments)
- `Discrete_HL7_Messages_Sample.txt` - a real-shaped GE ViewPoint vendor
  demo export (joke placeholder patient data - fake name, "Poor guy
  showed up to the ER" as the indication - confirmed not real PHI)

`tests/data/viewpoint_1-8.txt` is deliberately **not** used here - it's
gitignored real EVMS clinical narrative text, not synthetic demo data,
and has no place inside a committed notebook.

Saves each Phenopacket to `notebook_outputs/viewpoint/`.

In [ ]:
import gzip
from datetime import datetime, timezone
from pathlib import Path

from google.protobuf.json_format import MessageToJson, Parse
from google.protobuf.timestamp_pb2 import Timestamp
import phenopackets.schema.v2 as pps2

from prenatalppkt.builders import build_viewpoint_phenopacket
from prenatalppkt.etl.extractors import viewpoint_hl7
from prenatalppkt.genomics import (
    build_genomic_interpretation,
    build_vcf_file_entry,
    scan_vcf_file,
)
from prenatalppkt.hpo import HpoParser

print("=" * 80)
print("ViewPoint HL7 -> Phenopacket v2.0")
print("Covers every synthetic ViewPoint HL7 fixture: single fetus, twins, a full")
print("exam (biometry + all 16 anatomy fields), anatomy-only, and a real-shaped")
print("vendor demo export (Discrete_HL7_Messages_Sample.txt) - HPO features (the")
print("builder), LOINC-coded raw measurements, and dummy genomics scaffolding.")
print("=" * 80)

# -----------------------------------------------------------------------------
# STEP 1: Load the fenominal HPO Concept Recognizer (text -> HPO, detects negation)
# -----------------------------------------------------------------------------
HP_JSON_GZ = Path("tests/data/hp.json.gz")
TMP_HP_JSON = Path("/tmp/hp_viewpoint_cell.json")
with gzip.open(HP_JSON_GZ, "rt", encoding="utf-8") as f_in:
    with open(TMP_HP_JSON, "w", encoding="utf-8") as f_out:
        f_out.write(f_in.read())
hpo_parser = HpoParser(hpo_json_file=str(TMP_HP_JSON))
print(f"HPO version: {hpo_parser.get_version()}")

DATA_DIR = Path("tests/data")
now_ts = Timestamp()
now_ts.FromDatetime(datetime.now(tz=timezone.utc))

# Same dummy VCF reused across every fetus below (not tied to any one case) -
# see the matching note in the Observer cell above.
DUMMY_VCF = DATA_DIR / "Apple_Sally.vcf"

# Real EVMS export text (viewpoint_1-8.txt) is deliberately excluded here -
# it's gitignored real clinical narrative, not synthetic demo data, and has
# no place inside a committed notebook. Discrete_HL7_Messages_Sample.txt is
# a GE ViewPoint vendor sample with joke placeholder patient data (fake
# name, "Poor guy showed up to the ER" as the indication) - safe to use.
HL7_FILES = [
    "viewpoint_hl7_test.txt",
    "viewpoint_hl7_twins_test.txt",
    "viewpoint_hl7_full_exam_test.txt",
    "viewpoint_hl7_anatomy_test.txt",
    "Discrete_HL7_Messages_Sample.txt",
]
# Short, made-up accession ids per file - just labels for the demo, not a
# real accession-numbering scheme.
ACCESSION_BY_FILE = {
    "viewpoint_hl7_test.txt": "DEMO001",
    "viewpoint_hl7_twins_test.txt": "DEMOTWIN",
    "viewpoint_hl7_full_exam_test.txt": "DEMOFULL",
    "viewpoint_hl7_anatomy_test.txt": "DEMOANAT",
    "Discrete_HL7_Messages_Sample.txt": "DEMODISC",
}

OUTPUT_DIR = Path("notebook_outputs/viewpoint")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nFiles: {len(HL7_FILES)}")

for filename in HL7_FILES:
    print("\n" + "-" * 80)
    print(f"Processing: {filename}")
    print("-" * 80)

    data = (DATA_DIR / filename).read_text()
    accession_id = ACCESSION_BY_FILE[filename]

    # STEP 2: the builder does the real work - biometry, clinical impression,
    # fetal anatomy, dating all stitched into one Phenopacket per fetus.
    pps = build_viewpoint_phenopacket(data, hpo_parser, now_ts, accession_id=accession_id)
    print(f"Fetuses: {len(pps)}")

    # Re-run the extractor directly for the raw TermBins (LOINC code + mm
    # value) - the builder itself only emits phenotypicFeatures, same gap
    # as the Observer side.
    term_bins_by_fetus = viewpoint_hl7.extract_all_fetuses(data)

    for pp in pps:
        fetus_number = int(pp.id.rsplit("-", 1)[-1])
        term_bins = term_bins_by_fetus.get(fetus_number, [])

        # STEP 3: hand-add LOINC-coded Measurements alongside the HPO features.
        measurements = [
            pps2.Measurement(
                assay=pps2.OntologyClass(id=tb.loinc_code, label=tb.loinc_label),
                value=pps2.Value(
                    quantity=pps2.Quantity(
                        unit=pps2.OntologyClass(id="UO:0000016", label="millimeter"),
                        value=tb.value_mm,
                    )
                ),
            )
            for tb in term_bins
            if tb.loinc_code
        ]
        pp.measurements.extend(measurements)

        # STEP 4: attach dummy genomics scaffolding (VCF file + interpretation).
        # Structural only - no VRS normalization, no ACMG calls, no real linkage
        # between these variants and this fetus's actual phenotype.
        variants = scan_vcf_file(DUMMY_VCF)
        pp.files.append(
            build_vcf_file_entry(
                DUMMY_VCF.resolve().as_uri(),
                attributes={"genomeAssembly": variants[0].genome_assembly},
            )
        )
        pp.interpretations.append(
            build_genomic_interpretation(
                variants,
                subject_id=pp.subject.id,
                interpretation_id=f"{pp.id}-genomic-interp-1",
            )
        )
        print(
            f"  {pp.id}: {len(pp.phenotypic_features)} features, "
            f"{len(measurements)} measurements, {len(variants)} dummy variants"
        )

        # STEP 5: round-trip through JSON to confirm the Phenopacket is schema-valid
        json_str = MessageToJson(pp)
        round_tripped = Parse(json_str, pps2.Phenopacket())
        assert round_tripped == pp

        out_path = OUTPUT_DIR / f"{pp.id}_phenopacket.json"
        out_path.write_text(json_str)

print("\nSaved to:", OUTPUT_DIR)
TMP_HP_JSON.unlink(missing_ok=True)
